In [ ]:
!pip install Finance-Datareader  
#import FinanceDataReader as fdr

In [ ]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. 주식 데이터 수집
# -----------------------------
names = ['AAPL', 'TSLA', 'META', 'NVDA']

stocks = fdr.DataReader(names).dropna()

# -----------------------------
# 2. 일간 수익률 계산
# -----------------------------
daily_ret = stocks.pct_change().dropna()

# 연간 기대 수익률
annual_ret = daily_ret.mean() * 252

# 연간 공분산 행렬
annual_cov = daily_ret.cov() * 252

# -----------------------------
# 3. 포트폴리오 시뮬레이션
# -----------------------------
ret_box = []
risk_box = []
weights_box = []
sharpe_box = []

# 무위험 수익률
rf = 0.03

for i in range(30000):

    # 랜덤 비중 생성
    weights = np.random.random(len(names))
    weights = weights / np.sum(weights)

    # 기대 수익률
    returns = np.dot(weights, annual_ret)

    # 리스크(표준편차)
    risk = np.sqrt(np.dot(weights.T,
                          np.dot(annual_cov, weights)))

    # 샤프지수
    sharpe = (returns - rf) / risk

    # 저장
    ret_box.append(returns)
    risk_box.append(risk)
    weights_box.append(weights)
    sharpe_box.append(sharpe)

# -----------------------------
# 4. 데이터프레임 생성
# -----------------------------
result = pd.DataFrame(weights_box, columns=names)

result['Return'] = ret_box
result['Risk'] = risk_box
result['Sharpe'] = sharpe_box

# -----------------------------
# 5. 최대 샤프지수 포트폴리오
# -----------------------------
max_sharpe = result.loc[result['Sharpe'].idxmax()]

# -----------------------------
# 6. 최소 위험 포트폴리오
# -----------------------------
min_risk = result.loc[result['Risk'].idxmin()]

# -----------------------------
# 7. 결과 출력
# -----------------------------
print("===== 최대 샤프지수 포트폴리오 =====")
print(max_sharpe)

print("\n===== 최소 위험 포트폴리오 =====")
print(min_risk)

# -----------------------------
# 8. 시각화
# -----------------------------
plt.figure(figsize=(12,8))

scatter = plt.scatter(
    result['Risk'],
    result['Return'],
    c=result['Sharpe'],
    cmap='viridis',
    alpha=0.5
)

plt.colorbar(scatter, label='Sharpe Ratio')

# 최대 샤프지수 표시
plt.scatter(
    max_sharpe['Risk'],
    max_sharpe['Return'],
    c='red',
    marker='*',
    s=500,
    label='Max Sharpe'
)

# 최소 위험 표시
plt.scatter(
    min_risk['Risk'],
    min_risk['Return'],
    c='blue',
    marker='X',
    s=300,
    label='Min Risk'
)

plt.title('Portfolio Optimization')
plt.xlabel('Risk')
plt.ylabel('Expected Return')

plt.legend()

plt.show()